In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage,HumanMessage
from scripts.regsetup import description

load_dotenv()

model =  init_chat_model(model="deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

#1.准备工具
@tool
def get_weather(location:str) -> str:
    """根据location查询天气"""
    return f"{location}的天气是晴天"
@tool
def square_calc(x:float) -> float:
    """计算x的平方"""
    return x**2

#2.创建智能体
agent = create_agent(model = model,tools=[get_weather,square_calc])

#3.调用智能体，得到dict字典，其中有键messages是[聊天记录]，模型本次回复的消息是列表中最后一个
response = agent.invoke({
    "messages":[HumanMessage(content="深圳天气怎么样")]
})
print(type(response))
print(response['messages'][-1].content)



In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage,HumanMessage
load_dotenv()
model = init_chat_model(model = "deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

@tool
def get_weather(location:str) -> str:
    """根据location查询天气"""
    return f"{location}的天气是晴天"
agent = create_agent(model=model,tools=[get_weather],system_prompt="你是抖音主播小团团，你要学她一样说话")

response = agent.invoke({
    "messages":[HumanMessage(content="深圳天气怎么用")]
})
print(response["messages"][-1].content)

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

from langchain_core.messages import SystemMessage,HumanMessage
load_dotenv()
model = init_chat_model(model = "deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

agent = create_agent(model = model ,tools = [],system_prompt="你是一个资深的科幻小说作家，可以参考刘慈欣")
#调用智能体。注意：
#调用stream_events(input=消息字典，version="v3")方法时要传2个参数
#得到返回值是stream对象，其中包含了非常多的数据，但是我们目前只需要其中的消息数据
stream = agent.stream_events(
    input = {"messages":[HumanMessage(content="基于AI对现实的冲击，写一篇2000字以内的短篇小说")]},
    version = "v3"
)
#得到消息流：stream.messages，得到的是多条消息的流。因为和模型可能要交互多次，每次消息交互都是一个流
for message_stream in stream.messages:
    #每个消息流，通过消息流可以获取消息中的每个字符，打印出来
    for text in message_stream.text:
        #flush:表示以流式输出
        print(text,end="",flush=True)


In [6]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage,HumanMessage
load_dotenv()
model = init_chat_model(model = "deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

#工具注释的两种方法，实际上企业内可混合使用
@tool("get_weather",description = "获取城市的天气信息")
def get_weather(location : str,date : str) -> str:
    """
    查询天气
    Args:
        :param location:当前城市
               date : 时间
        :return:天气信息
    """
    return f"{date}  {location}的天气是晴天"
agent = create_agent(model = model,tools=[get_weather])
response = agent.invoke(
    {"messages":[HumanMessage(content="深圳现在的天气怎么样？")]}
)
print(response)

{'messages': [HumanMessage(content='深圳现在的天气怎么样？', additional_kwargs={}, response_metadata={}, id='665a26f1-9eeb-44fa-a2f4-42012570f5c3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 280, 'total_tokens': 334, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 152}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '99efdf46-0d88-4537-9542-8e5f9a21c2cc', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1867f-1e57-77ea-a2da-1f8fa527f0dc-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '深圳', 'date': '今天'}, 'id': 'call_00_wlHZS3uD4Os5pxL1BPi56518', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [12]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pydantic import Field,BaseModel
from datetime import datetime

from langchain_core.messages import SystemMessage,HumanMessage
load_dotenv()
model = init_chat_model(model = "deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

class WeatherInfo(BaseModel):
    location:str = Field(description= "城市",default="深圳")
    date:str = Field(description = "现在的日期时间",default = str(datetime.now().strftime("%Y-%m-%d, %H:%M:%S")))

@tool("get_weather",description="获取当前天气信息",args_schema=WeatherInfo)
def get_weather(location:str,date:str)->str:
    return f"{location}在{date}时的天气的雷暴雨"

agent = create_agent(model = model ,tools = [get_weather],system_prompt="你是天气预报员，尽可能提供详细的天气信息以及出行建议")
#调用智能体注意：
#调用stream_events(input=消息字典，version="v3")方法时要传2个参数
#得到返回值是stream对象，其中包含了非常多的数据，只需要其中的消息数据
stream = agent.stream_events(
    input = {"messages":[HumanMessage(content="深圳现在的天气如何呢")]},
    version = "v3"
)
#得到消息流：stream.messages，得到的是多条消息的流。因为和模型可能要交互多次，每次消息交互都是一个流
for message_stream in stream.messages:
    #每个消息流，通过消息流可以获取消息中的每个字符，打印出来
    for text in message_stream.text:
        #flush:表示以流式输出
        print(text,end="",flush=True)


我来帮您查询深圳当前的天气情况。# 🌩️ 深圳实时天气播报

**查询时间：2026年10月29日 15:39**

---

## 📢 当前天气状况

| 项目 | 情况 |
|------|------|
| **天气现象** | ⛈️ **雷暴雨** |
| **地点** | 广东·深圳 |
| **时段** | 下午（15:39） |

---

## ⚠️ 重点提示

当前深圳正经历**雷暴雨**天气，属于强对流天气，需特别注意以下几点：

### 🚨 安全出行建议

1. **尽量减少外出** 🏠
   - 雷暴雨期间请尽量留在室内，避免不必要的外出
   - 如已在户外，尽快寻找坚固建筑物躲避

2. **防雷击** ⚡
   - 远离大树、广告牌、电线杆、金属结构等高耸物体
   - 不要在水边、空旷地带停留
   - 避免使用金属雨伞，不要在空旷处打电话

3. **驾车出行** 🚗
   - 降低车速，开启雾灯和危险报警闪光灯
   - 保持安全车距，切勿急刹车
   - 遇到积水路段，先判断深度，**不要贸然涉水**，防止车辆熄火被困
   - 注意路面湿滑，谨防打滑侧翻

4. **步行注意** 🚶
   - 避开积水较深的路段和井盖缺失处
   - 远离临时围挡、老旧建筑，防范高空坠物
   - 留意下水道反涌和触电隐患（如倒伏电线杆）

5. **居家防护** 🏡
   - 关好门窗，收好阳台衣物、花盆等易被吹落物品
   - 拔掉不必要的电器插头，防范雷击损坏电器
   - 检查排水口是否通畅，防止积水倒灌

---

## 📱 温馨提示

> 雷暴雨天气变化较快，建议您关注深圳市气象局最新发布的**暴雨/雷电预警信号**（黄/橙/红），合理安排出行计划。若预警升级为橙色或红色，请务必停止户外活动。

---

需要我帮您查询**未来几天的天气趋势**或**其他城市**的天气情况吗？随时告诉我 😊